In [1]:
import os
import sys

# เพิ่ม Path ป้องกันการหาโฟลเดอร์ไม่เจอเมื่อรันใน Notebook
sys.path.append(os.path.abspath('./'))

from config.constants import AppConfig
from utils.logger import SystemLogger
from core.data_manager import DataManager
from core.api_client import CounterServiceClient
from core.flow_manager import RequestFlowManager
from core.database import OracleDBManager
from core.excel_manager import ExcelReportManager
from core.data_models import TransactionData
SystemLogger.show("โหลดไลบรารีระดับองค์กรสำเร็จ พร้อมทำงาน!")

[SYSTEM INFO] โหลดไลบรารีระดับองค์กรสำเร็จ พร้อมทำงาน!


In [2]:
# 1. กำหนดข้อมูลที่บังคับต้องมี (Required)
req_data = TransactionData(store_id="09892",vendor_id="0993000134168",service_id="02",item_name= "บริจาค",bill_amt=50,data_1='1234567890')



# 3. ให้ DataManager ประกอบร่างข้อมูล
data_mgr = DataManager(required_data=req_data)
final_payload = data_mgr.get_final_payload()

SystemLogger.show("ข้อมูล Payload พร้อมส่ง:")
# SystemLogger.display(final_payload)

[SYSTEM INFO] ข้อมูล Payload พร้อมส่ง:


In [4]:
from typing import Dict, Any, List

def process_transaction_sequence(v5_raw: Any, remove_text: str) -> Dict[str, Any]:
    """
    ฟังก์ชันสำหรับจัดการ String ของ Transaction ตัดข้อความ แยกด้วย '|' 
    และคำนวณ Sequence ใหม่แบบ 5 หลัก
    """
    # 1. รับค่าและแทนที่ข้อความ
    raw_tx_id = str(v5_raw)
    cleaned_str = raw_tx_id.replace(remove_text, "")
    
    # 2. แยกข้อมูลด้วย "|" (ถ้าไม่มี "|" คำสั่ง split จะได้ list 1 item อัตโนมัติ)
    tx_id_list = cleaned_str.split("|")
    list_length = len(tx_id_list)
    
    # 3. เตรียม List สำหรับเก็บค่าที่ประมวลผลแล้ว
    commontran_list: List[str] = []
    
    for item in tx_id_list:
        # ตัดตัวอักษร 3 หลักแรกออก
        cut_item = item[3:]
        
        try:
            # แปลงเป็น Int + จำนวนข้อมูลใน list
            calc_val = int(cut_item) + list_length
        except ValueError:
            # ดักจับ Error กรณีตัด 3 ตัวแรกแล้วไม่มีตัวเลขเหลือ หรือเป็นตัวอักษร
            calc_val = 0 
            
        # แปลงกลับเป็น String 5 หลัก เติม 0 ด้านหน้า
        formatted_val = str(calc_val).zfill(5)
        commontran_list.append(formatted_val)
        
    # 4. สร้าง Dictionary คืนค่าตามโครงสร้างที่ต้องการ
    result_dict = {
        "tx_id": raw_tx_id,
        "tx_id_list": tx_id_list,
        "commontran_list": commontran_list,
        # ใช้ join เพื่อต่อ String ด้วย "|" (ถ้ามีค่าเดียว join จะไม่เติม "|")
        "commontran": "|".join(commontran_list) 
    }
    
    return result_dict

# ==========================================
# ตัวอย่างการนำไปเรียกใช้งานในคลาสของคุณ:
# ==========================================
# สมมติว่า data.getdata() คืนค่า "146"
# remove_val = SystemConfig.get_data()  
#
# กรณีที่มี "|": 
# v[5] = "146ABC0012|146XYZ0098"
# output = process_transaction_sequence(v[5], remove_val)
#
# กรณีที่ไม่มี "|":
# v[5] = "146ABC0012"
# output = process_transaction_sequence(v[5], remove_val)

In [3]:
SystemLogger.show("เริ่มกระบวนการยิง API Request Flow...")

api_client = CounterServiceClient()
flow = RequestFlowManager(api_client)

# กำหนดลำดับการทำงานที่ต้องการให้โปรแกรมรันต่อเนื่อง
action_list = ["exchange", "reprint", "save", "confirm"]

# รัน Pipeline ทั้งหมดในคำสั่งเดียว
result_data = flow.run_pipeline(action_list, final_payload)
SystemLogger.show("สิ้นสุดการทำงาน Action Pipeline! ผลลัพธ์ที่ได้คืนมา:")
SystemLogger.display(result_data)
SystemLogger.display(flow.log_history())



[SYSTEM INFO] เริ่มกระบวนการยิง API Request Flow...
[SYSTEM INFO] Executing Chain Action: exchange
[SYSTEM INFO] Executing Chain Action: reprint
[SYSTEM INFO] Executing State Cache: save
[SYSTEM INFO] Executing Chain Action: confirm
[SYSTEM INFO] สิ้นสุดการทำงาน Action Pipeline! ผลลัพธ์ที่ได้คืนมา:
{   'action': 'confirm',
    'raw_response': '<?xml version="1.0" '
                    'encoding="UTF-8"?><HQ_RESPONSE><SUCCESS>false</SUCCESS><CODE>119</CODE><DESCRIPTOR>ขออภัยปิดปรับปรุง '
                    '30 '
                    'นาที</DESCRIPTOR><VENDOR_ID></VENDOR_ID><SERV_ID></SERV_ID><TX_ID></TX_ID><PRINTSLIP></PRINTSLIP><VAT></VAT><BILL_AMT></BILL_AMT><FEE></FEE><FEE_VAT></FEE_VAT><DATA_1></DATA_1><DATA_2></DATA_2><DATA_3></DATA_3><DATA_4></DATA_4><DATA_5></DATA_5><DATA_6></DATA_6><DATA_7></DATA_7><CUSTOMER_NAME></CUSTOMER_NAME><CUSTOMER_ADDR_1></CUSTOMER_ADDR_1><CUSTOMER_ADDR_2></CUSTOMER_ADDR_2><CUSTOMER_ADDR_3></CUSTOMER_ADDR_3><CUSTOMER_TEL_NO></CUSTOMER_TEL_NO><ACCT_NO></A

In [ ]:
SystemLogger.show("เริ่มเชื่อมต่อและตรวจสอบฐานข้อมูล Oracle...")

# คำสั่ง Query แบบ Dynamic (ไม่ต้อง Hardcode VENDOR_ID ใน SQL)
query = """
    SELECT VENDOR_CODE, VENDOR_NAME, SERVICE_ID, SYSTEM_TYPE 
    FROM ONLSTD.WS_CLIENT_CONFIG 
    WHERE VENDOR_ID = :vendor_id AND SERVICE_ID = :service_id
"""
params = {
    "vendor_id": final_payload["VENDOR_ID"], 
    "service_id": final_payload["SERVICE_ID"]
}

try:
    with OracleDBManager(host=AppConfig.DB_HOST_12C) as db:
        cols, rows = db.execute_query(query, params)
        if rows:
            # นำข้อมูลจับคู่เป็น Dictionary ให้อ่านง่าย
            db_result = dict(zip(cols, rows[0]))
            SystemLogger.display(db_result)
        else:
            SystemLogger.show("ไม่พบข้อมูลที่ตรงกับเงื่อนไขใน Database")
except Exception as e:
    SystemLogger.error_traceback(e)

In [ ]:
SystemLogger.show("กำลังสร้างและจัดรูปแบบไฟล์ Excel Report...")

excel_mgr = ExcelReportManager()

# 1. บันทึกข้อมูล Payload ลงหน้า Transaction
excel_mgr.write_data_to_sheet(AppConfig.SHEET_TXN, [final_payload])

# 2. บันทึกสถานะการรันลงหน้า Log
log_data = [{
    "Action": "Execute Pipeline", 
    "Status": "Success", 
    "Target_Vendor": final_payload["VENDOR_ID"],
    "Details": str(result_data)
}]
excel_mgr.write_data_to_sheet(AppConfig.SHEET_LOG, log_data)

# 3. เซฟไฟล์ลงเครื่อง
export_filename = f"Execution_Result_{final_payload['VENDOR_ID']}_{final_payload['SERVICE_ID']}.xlsx"
excel_mgr.save_to_file(export_filename)

SystemLogger.show(f"บันทึกไฟล์ {export_filename} สำเร็จเรียบร้อย สามารถเปิดดูไฟล์ได้ทันที!")